In [1]:
# Error analysis of the three Task A models on both test sets, using their saved predictions (no training).
# Descriptive only: nothing here changes models, thresholds or labels, and no prompt text is displayed.
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import transformers
from transformers import AutoTokenizer

# Find the repository root (works from any folder) and import the shared project code.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/task_a").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from task_a.cue_audit import CUES, review_priority

C:\Users\ONYX\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset v2 and the latest saved test predictions of each model (one column of 0/1 predictions per model).
df = pd.read_parquet(ROOT / "data/processed/binary_jailbreak_v2/combined.parquet",
                     columns=["example_id", "text", "label", "split", "source_dataset", "review_status"])
test = df[df["split"].isin(["test_official", "test_jailbreakhub"])].set_index("example_id")
model_folders = {"LogReg": "results/task_a_binary_tfidf_v2", "SVM": "results/task_a_binary_tfidf_linear_svm_v2",
                 "DistilBERT": "results/task_a_binary_distilbert_v2"}
models = list(model_folders)
for model, folder in model_folders.items():
    prediction_file = sorted((ROOT / folder).glob("run_*/predictions.csv"))[-1]
    saved = pd.read_csv(prediction_file).set_index("example_id")
    test[model] = saved["pred"]
    assert (saved.loc[test.index, "gold"] == test["label"]).all(), f"{model} gold labels differ from the dataset"
    print(f"{model}: {prediction_file.parent.name}")
assert test[models].notna().all().all(), "a test prompt has no saved prediction"

LogReg: run_20261021T022858Z
SVM: run_20261021T022904Z
DistilBERT: run_20261021T024649Z


In [3]:
# Error counts per model and test set: missed attacks (false negatives) and benign prompts flagged (false positives).
split_names = {"test_official": "Official WildGuardMix test", "test_jailbreakhub": "JailbreakHub test"}
counts = {}
for split, name in split_names.items():
    part = test[test["split"] == split]
    for model in models:
        counts[(name, model)] = {"Attacks": (part["label"] == 1).sum(), "Missed attacks": ((part["label"] == 1) & (part[model] == 0)).sum(),
                                 "Benign": (part["label"] == 0).sum(), "Benign flagged": ((part["label"] == 0) & (part[model] == 1)).sum()}
pd.DataFrame(counts).T

Attacks  Missed attacks  Benign  \
Official WildGuardMix test LogReg          810               3     915   
                           SVM             810               2     915   
                           DistilBERT      810               3     915   
JailbreakHub test          LogReg          141              47    1306   
                           SVM             141              43    1306   
                           DistilBERT      141              65    1306   

                                       Benign flagged  
Official WildGuardMix test LogReg                   8  
                           SVM                      6  
                           DistilBERT               1  
JailbreakHub test          LogReg                  64  
                           SVM                     51  
                           DistilBERT              16

In [4]:
# How many of the three models got each prompt wrong: errors shared by all three are the hardest cases.
test["models_wrong"] = test[models].ne(test["label"], axis=0).sum(axis=1)
pd.crosstab([test["split"], test["label"].map({0: "Benign", 1: "Attack"})], test["models_wrong"]).rename(columns=lambda n: f"wrong by {n}")

models_wrong              wrong by 0  wrong by 1  wrong by 2  wrong by 3
split             label                                                 
test_jailbreakhub Attack          69          26           9          37
                  Benign        1234          23          39          10
test_official     Attack         805           2           3           0
                  Benign         907           2           5           1

In [5]:
# Prompt features: DistilBERT token count (including [CLS]/[SEP]) and whether it is cut at 256 tokens.
transformers.logging.set_verbosity_error()  # silence the "longer than 512" notice: we only count tokens here
tokenizer = AutoTokenizer.from_pretrained(sorted((ROOT / model_folders["DistilBERT"]).glob("run_*/model"))[-1])
test["tokens"] = [len(ids) for ids in tokenizer(test["text"].tolist(), truncation=False)["input_ids"]]
test["over_256"] = test["tokens"] > 256

In [6]:
# Position of the first strong jailbreak cue (cue_audit patterns, AIPRM templates excluded), in tokens.
# With 256-token inputs, a cue starting at token 255 or later is cut off before DistilBERT sees it.
def first_strong_cue_token(text):
    _, strong_categories, _ = review_priority(text)
    starts = [CUES[category][0].search(text).start() for category in strong_categories]
    return len(tokenizer(text[:min(starts)])["input_ids"]) - 1 if starts else np.nan  # tokens before the cue

test["first_cue_token"] = test["text"].map(first_strong_cue_token)
test["cue_after_cutoff"] = test["first_cue_token"] >= 255

In [7]:
# Familiarity: TF-IDF cosine similarity to the closest training prompt (baseline TF-IDF settings).
# A low value means the prompt is worded unlike anything in training, e.g. an unfamiliar attack formulation.
train_text = df.loc[df["split"] == "train", "text"]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(train_text)
train_matrix, test_matrix = vectorizer.transform(train_text), vectorizer.transform(test["text"])
closest = []
for start in range(0, test_matrix.shape[0], 500):  # in chunks to limit memory
    closest.append((test_matrix[start:start + 500] @ train_matrix.T).max(axis=1).toarray().ravel())
test["closest_train_similarity"] = np.concatenate(closest)

In [8]:
# Profile of a group of prompts, used in the next three tables.
def profile(group):
    return {"Prompts": len(group), "Median tokens": group["tokens"].median(), "Over 256 tokens": group["over_256"].mean(),
            "No strong cue": group["first_cue_token"].isna().mean(), "Cue after token 255": group["cue_after_cutoff"].mean(),
            "Median similarity to training": group["closest_train_similarity"].median()}

table_format = {"Prompts": "{:,.0f}", "Median tokens": "{:,.0f}", "Over 256 tokens": "{:.0%}", "No strong cue": "{:.0%}",
                "Cue after token 255": "{:.0%}", "Median similarity to training": "{:.2f}"}

In [9]:
# JailbreakHub attacks: are the missed ones longer, cut at 256, cue-late, or less similar to training prompts?
jbh_attacks = test[(test["split"] == "test_jailbreakhub") & (test["label"] == 1)]
attack_profile = pd.DataFrame({(model, outcome): profile(jbh_attacks[jbh_attacks[model] == prediction])
                               for model in models for outcome, prediction in [("missed", 0), ("caught", 1)]}).T
attack_profile.style.format(table_format)

In [10]:
# JailbreakHub benign prompts: how do wrongly flagged ones differ from correctly passed ones?
jbh_benign = test[(test["split"] == "test_jailbreakhub") & (test["label"] == 0)]
benign_profile = pd.DataFrame({(model, outcome): profile(jbh_benign[jbh_benign[model] == prediction])
                               for model in models for outcome, prediction in [("flagged", 1), ("passed", 0)]}).T
benign_profile.style.format(table_format)

In [11]:
# Differences between the two test sources, per label (all prompts, regardless of model errors).
source_profile = pd.DataFrame({(split_names[split], "Attack" if label else "Benign"): profile(group)
                               for (split, label), group in test.groupby(["split", "label"])}).T
source_profile.style.format(table_format)

In [12]:
# Every official-test error (few), features only: which models erred and the prompt profile.
official_errors = test[(test["split"] == "test_official") & (test["models_wrong"] > 0)]
official_errors[["label", *models, "tokens", "first_cue_token", "closest_train_similarity"]].sort_values("label")

,label,LogReg,SVM,DistilBERT,tokens,first_cue_token,closest_train_similarity
example_id,,,,,,,
wildguard:test:167,0,1,1,0,65,NaN,0.143150
wildguard:test:210,0,1,0,0,47,NaN,0.247104
wildguard:test:217,0,1,0,0,33,NaN,0.366770
wildguard:test:265,0,1,1,0,65,NaN,0.183908
wildguard:test:286,0,1,1,1,71,NaN,0.134709
wildguard:test:284,0,1,1,0,71,NaN,0.150407
wildguard:test:814,0,1,1,0,22,NaN,0.249151
wildguard:test:858,0,1,1,0,67,NaN,0.449648
wildguard:test:82,1,0,1,0,16,NaN,0.538775


In [13]:
# Label-review candidates (NOT corrections): attacks missed by all three with no strong cue, and benign prompts
# flagged by all three that contain a strong cue. Text goes to a Git-ignored CSV for private reading, not the screen.
candidates = test[((test["label"] == 1) & (test["models_wrong"] == 3) & test["first_cue_token"].isna())
                  | ((test["label"] == 0) & (test["models_wrong"] == 3) & test["first_cue_token"].notna())]
out_dir = ROOT / "results/error_analysis" / datetime.now(timezone.utc).strftime("task_a_v2_%Y%m%dT%H%M%SZ")
out_dir.mkdir(parents=True, exist_ok=False)
candidates.reset_index()[["example_id", "split", "label", "review_status", "tokens", "first_cue_token",
                          "closest_train_similarity", "text"]].to_csv(out_dir / "label_review_candidates.csv", index=False)
test.drop(columns="text").to_csv(out_dir / "test_error_features.csv")
print(f"{len(candidates)} candidates ({candidates['review_status'].ne('').sum()} already reviewed earlier); "
      f"saved to {out_dir.relative_to(ROOT)}")
pd.crosstab(candidates["split"], candidates["label"].map({0: "Benign flagged by all 3", 1: "Attack missed by all 3"}))

38 candidates (1 already reviewed earlier); saved to results\error_analysis\task_a_v2_20261021T203005Z


label,Attack missed by all 3,Benign flagged by all 3
split,,
test_jailbreakhub,31,7


In [14]:
# Summary generated from the tables above. These are associations within one test set, not proven causes.
for model in models:
    missed, caught = attack_profile.loc[(model, "missed")], attack_profile.loc[(model, "caught")]
    print(f"{model}: missed {missed['Prompts']:.0f} JailbreakHub attacks. Missed vs caught: median {missed['Median tokens']:.0f} vs "
          f"{caught['Median tokens']:.0f} tokens, over 256 tokens {missed['Over 256 tokens']:.0%} vs {caught['Over 256 tokens']:.0%}, "
          f"no strong cue {missed['No strong cue']:.0%} vs {caught['No strong cue']:.0%}, "
          f"cue after token 255 {missed['Cue after token 255']:.0%} vs {caught['Cue after token 255']:.0%}, "
          f"similarity to training {missed['Median similarity to training']:.2f} vs {caught['Median similarity to training']:.2f}.")
all_missed = jbh_attacks[jbh_attacks["models_wrong"] == 3]
print(f"{len(all_missed)} of {len(jbh_attacks)} JailbreakHub attacks were missed by all three models; "
      f"{all_missed['first_cue_token'].isna().mean():.0%} of those contain no strong cue.")
print(f"Benign prompts flagged by all three models: {((test['label'] == 0) & (test['models_wrong'] == 3)).sum()} "
      f"(JailbreakHub {((jbh_benign['models_wrong'] == 3)).sum()}).")
print("Limitations: cue position is a proxy for where the malicious instruction sits; review candidates may be "
      "correct labels; JailbreakHub labels are partly unreviewed and the 174 corrections are not human-verified.")

LogReg: missed 47 JailbreakHub attacks. Missed vs caught: median 210 vs 504 tokens, over 256 tokens 47% vs 80%, no strong cue 74% vs 17%, cue after token 255 4% vs 10%, similarity to training 0.22 vs 0.38.
SVM: missed 43 JailbreakHub attacks. Missed vs caught: median 192 vs 454 tokens, over 256 tokens 47% vs 79%, no strong cue 74% vs 19%, cue after token 255 5% vs 9%, similarity to training 0.24 vs 0.38.
DistilBERT: missed 65 JailbreakHub attacks. Missed vs caught: median 300 vs 540 tokens, over 256 tokens 58% vs 78%, no strong cue 68% vs 9%, cue after token 255 8% vs 8%, similarity to training 0.24 vs 0.41.
37 of 141 JailbreakHub attacks were missed by all three models; 84% of those contain no strong cue.
Benign prompts flagged by all three models: 11 (JailbreakHub 10).
Limitations: cue position is a proxy for where the malicious instruction sits; review candidates may be correct labels; JailbreakHub labels are partly unreviewed and the 174 corrections are not human-verified.
